In [3]:
import torch
import torch.nn as nn
import pickle

In [4]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (h_n, _) = self.lstm(embedded)
        return self.fc(h_n[-1])

In [5]:
with open("word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

In [6]:
word2idx

{'<pad>': 0,
 '<unk>': 1,
 '太': 2,
 '了': 3,
 '演技': 4,
 '剧情': 5,
 '浪费': 6,
 '不': 7,
 '推荐': 8,
 '这个': 9,
 '电影': 10,
 '好看': 11,
 '演员': 12,
 '在线': 13,
 '精彩': 14,
 '非常': 15,
 '喜欢': 16,
 '这部': 17,
 '作品': 18,
 '画面': 19,
 '精美': 20,
 '值得': 21,
 '一看': 22,
 '时间': 23,
 '烂片': 24,
 '拖沓': 25,
 '尴尬': 26,
 '完全': 27,
 '看': 28,
 '下去': 29,
 '钱': 30,
 '这': 31,
 '片子': 32,
 '真': 33,
 '不错': 34,
 '故事': 35,
 '感人': 36,
 '强烈': 37,
 '烂': 38,
 '到': 39,
 '极点': 40,
 '别看': 41,
 '毫无': 42,
 '亮点': 43,
 '失望': 44}

In [8]:
model = torch.load("lstm_full_model.pt", weights_only=False)
model

LSTMClassifier(
  (embedding): Embedding(45, 32, padding_idx=0)
  (lstm): LSTM(32, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

In [9]:
model.eval()

LSTMClassifier(
  (embedding): Embedding(45, 32, padding_idx=0)
  (lstm): LSTM(32, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

In [10]:
def predict(text, model, word2idx):
    model.eval()
    tokens = text.split()
    indices = [word2idx.get(t, 1) for t in tokens]

    x = torch.tensor([indices], dtype=torch.long)

    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()

    labels = {0: "负面 😞", 1: "正面 😊"}
    return labels[pred], prob[0][pred].item()

In [11]:
test_data = [
    ("这个 电影 太 好看 了", 1),
    ("太 浪费 时间 了 烂片", 0),
    ("演员 演技 在线 剧情 精彩", 1),
    ("完全 看 不 下去", 0),
    ("非常 喜欢 这部 作品", 1),
    ("毫无 亮点 失望", 0),
]

In [12]:
correct = 0

print("预测结果：")
print("-" * 45)

for text, true_label in test_data:
    tokens = text.split()

    indices = [word2idx.get(t, 1) for t in tokens]

    x = torch.tensor([indices], dtype=torch.long)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=1).item()

    status = "正确" if pred == true_label else "错误"
    if pred == true_label:
        correct += 1

    print(f"  {text:<20} | 真实: {true_label} | 预测: {pred} | {status}")
print("-" * 45)

预测结果：
---------------------------------------------
  这个 电影 太 好看 了         | 真实: 1 | 预测: 1 | 正确
  太 浪费 时间 了 烂片         | 真实: 0 | 预测: 0 | 正确
  演员 演技 在线 剧情 精彩       | 真实: 1 | 预测: 1 | 正确
  完全 看 不 下去            | 真实: 0 | 预测: 0 | 正确
  非常 喜欢 这部 作品          | 真实: 1 | 预测: 1 | 正确
  毫无 亮点 失望             | 真实: 0 | 预测: 0 | 正确
---------------------------------------------


In [13]:
print(f"准确率: {correct}/{len(test_data)} = {correct/len(test_data):.0%}")

准确率: 6/6 = 100%


In [14]:
test_data = [
    ("这个 综艺 太 好看 了", 1),
    ("烂片", 0),
    ("演员 演技 在线", 1),
    ("看 不 下去", 0),
    ("非常 喜欢", 1),
    ("失望", 0),
]

correct = 0

print("预测结果：")
print("-" * 45)

for text, true_label in test_data:
    tokens = text.split()

    indices = [word2idx.get(t, 1) for t in tokens]

    x = torch.tensor([indices], dtype=torch.long)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=1).item()

    status = "正确" if pred == true_label else "错误"
    if pred == true_label:
        correct += 1

    print(f"  {text:<20} | 真实: {true_label} | 预测: {pred} | {status}")
print("-" * 45)
print(f"准确率: {correct}/{len(test_data)} = {correct/len(test_data):.0%}")

预测结果：
---------------------------------------------
  这个 综艺 太 好看 了         | 真实: 1 | 预测: 1 | 正确
  烂片                   | 真实: 0 | 预测: 0 | 正确
  演员 演技 在线             | 真实: 1 | 预测: 1 | 正确
  看 不 下去               | 真实: 0 | 预测: 0 | 正确
  非常 喜欢                | 真实: 1 | 预测: 1 | 正确
  失望                   | 真实: 0 | 预测: 0 | 正确
---------------------------------------------
准确率: 6/6 = 100%
